# exp013 ensemble model

## 0. Experiment Metadata

In [1]:
# ========================================
# EXPERIMENT CONFIG
# ========================================

EXP_NAME = "exp013_ensemble_model"
TARGET = "Churn"
ID_COL = "id"

N_SPLITS = 5
SEED = 42

print(f"Running {EXP_NAME}")

Running exp013_ensemble_model


## 1. Imports

In [2]:
# ========================================
# IMPORTS
# ========================================

import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',50)

## 2. Reproducibility

In [3]:
# ========================================
# SEED EVERYTHING
# ========================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

## 3. Load Data

In [4]:
# ========================================
# LOAD DATA
# ========================================

DATA_PATH = "/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/data/raw/"

train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

print(train.shape, test.shape)
train.head()

(594194, 21) (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


## 4. Quick Sanity Checs (Fast EDA)

In [5]:
# ========================================
# QUICK EDA
# ========================================

print("Target distribution:")
print(train['Churn'].value_counts(normalize=True))

print("\nMissing values (top 10):")
print(train.isnull().mean().sort_values(ascending=False).head(10))

Target distribution:
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64

Missing values (top 10):
id                  0.0
DeviceProtection    0.0
TotalCharges        0.0
MonthlyCharges      0.0
PaymentMethod       0.0
PaperlessBilling    0.0
Contract            0.0
StreamingMovies     0.0
StreamingTV         0.0
TechSupport         0.0
dtype: float64


## 5. Feature Engineering 

In [6]:
# ========================================
# FEATURE ENGINEERING (EXP006 - COMBINED BEST)
# ========================================

service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", 
    "TechSupport", "StreamingTV", "StreamingMovies"
]

def create_features(df):
    df = df.copy()

    # ========================================
    # FINANCIAL (STRONG SIGNAL)
    # ========================================
    if all(col in df.columns for col in ["TotalCharges", "tenure"]):
        df["avg_charge"] = df["TotalCharges"] / (df["tenure"] + 1)

    if all(col in df.columns for col in ["TotalCharges", "MonthlyCharges", "tenure"]):
        df["charge_diff"] = df["TotalCharges"] - (df["MonthlyCharges"] * df["tenure"])

    # ========================================
    # SERVICE USAGE (STRONG SIGNAL)
    # ========================================
    if all(col in df.columns for col in service_cols):
        df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)

    # ========================================
    # BUSINESS LOGIC (VERY STRONG)
    # ========================================
    if "PaymentMethod" in df.columns:
        df["payment_risk"] = (df["PaymentMethod"] == "Electronic check").astype(int)

    if "InternetService" in df.columns:
        df["is_fiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    # ========================================
    # OPTIONAL CLEANUP (IMPORTANT DECISION)
    # ========================================
    # Drop TotalCharges ONLY if using avg_charge + charge_diff
    # if "TotalCharges" in df.columns:
    #     df = df.drop(columns=["TotalCharges"])

    return df


# Apply
train = create_features(train)
test = create_features(test)

## 6. Target Encoding

In [7]:
# ========================================
# TARGET ENCODING (SIMPLE VERSION)
# ========================================

from category_encoders import TargetEncoder

# Get categorical columns from TRAIN
categorical_cols = train.select_dtypes(include="object").columns.tolist()

# Remove target column
categorical_cols = [col for col in categorical_cols if col != TARGET]

# Initialize encoder
encoder = TargetEncoder(cols=categorical_cols)

# Fit on train, transform both
train[categorical_cols] = encoder.fit_transform(train[categorical_cols], train[TARGET])
test[categorical_cols] = encoder.transform(test[categorical_cols])

## 7. Prepare Data

In [8]:
# ========================================
# PREPARE DATA
# ========================================

features = [col for col in train.columns if col not in [TARGET, ID_COL]]

train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1}).astype(int)

X = train[features]
y = train[TARGET]

X_test = test[features]

## 8. Validation Strategy

In [9]:
# ========================================
# CV SETUP
# ========================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

## 9. Model Training

In [21]:
# h2o.shutdown(prompt=False)

/var/folders/jq/hz26s5p53n59bz8_pplxgwl80000gn/T/ipykernel_11915/1954269801.py:1: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.
  h2o.shutdown(prompt=False)


In [20]:
import h2o
from h2o.automl import H2OAutoML

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.


H2OServerError: Cluster reports unhealthy status

In [22]:
# ========================================
# SEED EVERYTHING
# ========================================
def seed_everything(seed):
    import random, os
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


# ========================================
# CONFIG
# ========================================
SEEDS = [42, 52, 62]
N_SPLITS = 5

oof_preds_lgb = np.zeros(len(train))
oof_preds_xgb = np.zeros(len(train))

test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))


# ========================================
# MULTI-SEED TRAINING (LGB + XGB ONLY)
# ========================================
for seed in SEEDS:
    
    print(f"\n====================")
    print(f"SEED {seed}")
    print(f"====================")

    seed_everything(seed)

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    oof_lgb_seed = np.zeros(len(train))
    oof_xgb_seed = np.zeros(len(train))

    test_lgb_seed = np.zeros(len(test))
    test_xgb_seed = np.zeros(len(test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        
        print(f"\n--- Fold {fold} ---")

        X_train, X_valid = X.iloc[tr_idx], X.iloc[val_idx]
        y_train, y_valid = y.iloc[tr_idx], y.iloc[val_idx]

        # ======================
        # LIGHTGBM
        # ======================
        lgb_model = lgb.LGBMClassifier(
            n_estimators=5000,
            learning_rate=0.01,
            num_leaves=64,
            max_depth=-1,
            min_child_samples=30,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=0.5,
            random_state=seed,
            n_jobs=-1
        )

        lgb_model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
        )

        val_pred_lgb = lgb_model.predict_proba(X_valid)[:, 1]
        oof_lgb_seed[val_idx] = val_pred_lgb
        test_lgb_seed += lgb_model.predict_proba(X_test)[:, 1] / N_SPLITS


        # ======================
        # XGBOOST
        # ======================
        xgb_model = xgb.XGBClassifier(
            n_estimators=5000,
            learning_rate=0.01,
            max_depth=4,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=1,
            reg_alpha=0.5,
            reg_lambda=1.0,
            eval_metric="auc",
            random_state=seed,
            n_jobs=-1,
            tree_method="hist"
        )

        xgb_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

        val_pred_xgb = xgb_model.predict_proba(X_valid)[:, 1]
        oof_xgb_seed[val_idx] = val_pred_xgb
        test_xgb_seed += xgb_model.predict_proba(X_test)[:, 1] / N_SPLITS


    # ======================
    # ACCUMULATE SEEDS
    # ======================
    oof_preds_lgb += oof_lgb_seed / len(SEEDS)
    oof_preds_xgb += oof_xgb_seed / len(SEEDS)

    test_preds_lgb += test_lgb_seed / len(SEEDS)
    test_preds_xgb += test_xgb_seed / len(SEEDS)


# ========================================
# H2O TRAINING (RUN ONCE - FULL DATA)
# ========================================
import h2o
from h2o.automl import H2OAutoML

# restart clean
try:
    h2o.shutdown(prompt=False)
except:
    pass

h2o.init(max_mem_size="4G")

train_h2o = train.copy()
train_h2o["target"] = y

hf_train = h2o.H2OFrame(train_h2o)
hf_test = h2o.H2OFrame(X_test)

hf_train["target"] = hf_train["target"].asfactor()

features = [col for col in hf_train.columns if col != "target"]

aml = H2OAutoML(
    max_runtime_secs=300,
    nfolds=5,
    balance_classes=True,
    sort_metric="AUC",
    seed=42,
    include_algos=["GBM", "GLM", "DRF", "StackedEnsemble"],
    keep_cross_validation_predictions=True
)

aml.train(x=features, y="target", training_frame=hf_train)

best_model = aml.leader

# H2O predictions (NO OOF → use as test-only model)
test_preds_h2o = best_model.predict(hf_test).as_data_frame()["p1"].values


# ========================================
# CV SCORES
# ========================================
print("\nLGB CV:", roc_auc_score(y, oof_preds_lgb))
print("XGB CV:", roc_auc_score(y, oof_preds_xgb))


# ========================================
# ENSEMBLE (2-model optimized + H2O add)
# ========================================
best_score = 0
best_w = 0

# optimize LGB + XGB first
for w in np.arange(0.0, 1.01, 0.01):
    oof_comb = w * oof_preds_lgb + (1 - w) * oof_preds_xgb
    score = roc_auc_score(y, oof_comb)

    if score > best_score:
        best_score = score
        best_w = w

print(f"\nBest LGB weight: {best_w:.2f}")
print(f"Best CV (LGB+XGB): {best_score:.5f}")


# ========================================
# FINAL TEST ENSEMBLE (ADD H2O)
# ========================================
# simple safe blend (since no OOF for H2O)
test_preds = (
    0.45 * test_preds_lgb +
    0.45 * test_preds_xgb +
    0.10 * test_preds_h2o
)


SEED 42

--- Fold 0 ---
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004840 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1162
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2252]	valid_0's binary_logloss: 0.298413

--- Fold 1 ---
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004733 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

/var/folders/jq/hz26s5p53n59bz8_pplxgwl80000gn/T/ipykernel_11915/2338116725.py:127: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.
  h2o.shutdown(prompt=False)


.... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "25.0.2" 2026-01-20; OpenJDK Runtime Environment Homebrew (build 25.0.2); OpenJDK 64-Bit Server VM Homebrew (build 25.0.2, mixed mode, sharing)
  Starting server from /Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/venv/lib/python3.11/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /var/folders/jq/hz26s5p53n59bz8_pplxgwl80000gn/T/tmps0b78o7c
  JVM stdout: /var/folders/jq/hz26s5p53n59bz8_pplxgwl80000gn/T/tmps0b78o7c/h2o_theojeremiah_started_from_python.out
  JVM stderr: /var/folders/jq/hz26s5p53n59bz8_pplxgwl80000gn/T/tmps0b78o7c/h2o_theojeremiah_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,Asia/Jakarta
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,16 days
H2O_cluster_name:,H2O_from_python_theojeremiah_b4ftjm
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.981 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%


/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/venv/lib/python3.11/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"



LGB CV: 0.9165581709833943
XGB CV: 0.9167445747404263

Best LGB weight: 0.32
Best CV (LGB+XGB): 0.91679


#### prev score 0.91669

## 11. Feature Importance

In [113]:
# ========================================
# FEATURE IMPORTANCE
# ========================================

import matplotlib.pyplot as plt

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance.head(20)

plt.figure(figsize=(8, 10))
plt.barh(importance["feature"].head(20), importance["importance"].head(20))
plt.gca().invert_yaxis()
plt.title("Top Features")
plt.show()

ValueError: All arrays must be of the same length

## 12. Create Submission

In [23]:
# ========================================
# SUBMISSION (FINAL - SAFE VERSION)
# ========================================
import os
import pandas as pd
import numpy as np

# ========================================
# SAFETY CHECKS
# ========================================

# 1. Clip probabilities (avoid weird values)
test_preds = np.clip(test_preds, 0, 1)

# 2. Ensure correct length
assert len(test_preds) == len(test), "Mismatch between predictions and test rows"

# 3. Ensure ID exists
assert ID_COL in test.columns, f"{ID_COL} not found in test data"

# ========================================
# CREATE SUBMISSION
# ========================================
submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,   # preserve order
    TARGET: test_preds             # probability (correct for AUC)
})

# Optional: sort by ID (only if Kaggle requires)
# submission = submission.sort_values(by=ID_COL)

# ========================================
# SAVE FILE
# ========================================
OUTPUT_DIR = "/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/outputs/submissions/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

file_path = os.path.join(OUTPUT_DIR, f"{EXP_NAME}.csv")

submission.to_csv(file_path, index=False)

print(f"\n✅ Submission saved at: {file_path}")

# ========================================
# FINAL CHECK (VERY IMPORTANT)
# ========================================
print("\nSubmission preview:")
print(submission.head())

print("\nStats:")
print("Min prob:", submission[TARGET].min())
print("Max prob:", submission[TARGET].max())
print("Mean prob:", submission[TARGET].mean())


✅ Submission saved at: /Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/outputs/submissions/exp013_ensemble_model.csv

Submission preview:
       id     Churn
0  594194  0.063820
1  594195  0.000997
2  594196  0.110141
3  594197  0.003799
4  594198  0.504486

Stats:
Min prob: 0.0007047600002856016
Max prob: 0.9336958643044426
Mean prob: 0.21464987127910223
